# 31 – Repository + Service Patterns

## Learning objectives

- Use `AbstractRepository` and `InMemoryRepository` from the core package.
- Orchestrate business rules through `BaseService` lifecycle hooks.
- See how `AuditedService` stamps user fields on create/update.

## About this topic

Repositories isolate persistence; services own use-cases and cross-cutting rules. Hooks such as `before_create` / `after_update` keep controllers thin and make it easy to add logging, events, or authorization without touching storage code.

In [ ]:
import asyncio
from pydantic import Field

from agentic_assistants.core.base_models import AgenticEntity
from agentic_assistants.core.repository import InMemoryRepository

class NotebookTask(AgenticEntity):
    title: str = Field(min_length=1)
    status: str = "open"

async def demo_repo() -> None:
    repo = InMemoryRepository(NotebookTask)
    t = NotebookTask(title="Wire repository demo")
    await repo.create(t)
    got = await repo.get(t.id)
    print("stored:", got.model_dump() if got else None)

asyncio.run(demo_repo())

In [ ]:
import asyncio
from pydantic import Field

from agentic_assistants.core.base_models import AgenticEntity
from agentic_assistants.core.repository import InMemoryRepository
from agentic_assistants.core.service import BaseService

class Doc(AgenticEntity):
    name: str = Field(min_length=1)

class LoggingService(BaseService[Doc]):
    async def after_create(self, data: Doc) -> None:
        print("after_create:", data.name)

    async def before_delete(self, id) -> None:
        print("before_delete:", id)

async def main() -> None:
    repo = InMemoryRepository(Doc)
    svc = LoggingService(repo)
    d = await svc.create(Doc(name="hooks-demo"))
    await svc.delete(d.id)

asyncio.run(main())

In [ ]:
import asyncio
from pathlib import Path
import importlib.util

from pydantic import Field

from agentic_assistants.core.base_models import AgenticAuditEntity
from agentic_assistants.core.repository import InMemoryRepository
from agentic_assistants.core.service import AuditedService

class AuditedDoc(AgenticAuditEntity):
    name: str = Field(min_length=1)

async def main() -> None:
    repo = InMemoryRepository(AuditedDoc)
    svc = AuditedService(repo, current_user="notebook-user")
    doc = await svc.create(AuditedDoc(name="audit trail"))
    print("created_by:", doc.created_by)

asyncio.run(main())

from pathlib import Path
import importlib.util

def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd if (cwd / "pyproject.toml").exists() else cwd.parent

path = _repo_root() / "examples/library/advanced_alchemy_patterns/service_layer.py"
spec = importlib.util.spec_from_file_location("adv_svc", path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
mod.demo_service_layer()

## Exercises and next steps

1. Implement a `list` filter in `InMemoryRepository` that supports `FilterSpec` on `status` for `NotebookTask`.
2. Override `before_create` to reject titles containing profanity (use a small denylist).
3. Skim `examples/library/advanced_alchemy_patterns/generic_repository.py` and contrast it with `AbstractRepository` in this repo.